# Project: Arabic Bank Check Amount Extraction and Processing


## Google Colab Setup


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import ast
import glob
import os
import random
import re
import shutil
import subprocess

subprocess.run(["pip", "install", "-q", "ultralytics", "editdistance"], check=True)

DRIVE_ROOT = Path("/content/drive/MyDrive")
PROJECT_FOLDER_NAME = "Arabic-Bank-Check-Amount-Extraction-and-Processing"
PROJECT_DIR = DRIVE_ROOT / PROJECT_FOLDER_NAME

if not PROJECT_DIR.exists():
    matches = sorted(DRIVE_ROOT.glob("Arabic-Bank-Check-*"))
    if len(matches) == 1:
        PROJECT_DIR = matches[0]
    else:
        raise FileNotFoundError(
            f"Could not find {PROJECT_FOLDER_NAME!r} under {DRIVE_ROOT}. "
            "Update PROJECT_FOLDER_NAME to match the Google Drive folder name."
        )

# --- Training inputs ---
IMAGES_DIR = PROJECT_DIR / "Images"
BOUNDING_BOXES_DIR = PROJECT_DIR / "BoundingBoxes"
COURTESY_AMOUNTS_DIR = PROJECT_DIR / "CourtesyAmounts"
COURTESY_AMOUNTS_RAW_DIR = PROJECT_DIR / "CourtesyAmounts_raw"
LEGAL_AMOUNTS_RAW_TEXT_DIR = PROJECT_DIR / "LegalAmounts_raw_text"
LEGAL_AMOUNTS_TOKENIZED_DIR = PROJECT_DIR / "LegalAmounts"

# --- Test set (CENPARMI test split) ---
TEST_ROOT = PROJECT_DIR / "CheckImages-Test" / "CheckImages-Test"
TEST_ANNOT_ROOT = PROJECT_DIR / "CheckAnnotations-Test" / "CheckAnnotations-Test"
TEST_IMAGES_DIR = TEST_ROOT
TEST_BBOX_DIR = TEST_ANNOT_ROOT / "BoundingBox"
TEST_COURTESY_LABELS = TEST_ANNOT_ROOT / "CourtesyAmounts.txt"
TEST_LEGAL_LABELS = TEST_ANNOT_ROOT / "LegalAmounts.txt"

# --- Working directories ---
WORK_DIR = Path("/content/arabic_bank_check_work")
YOLO_DATASET_DIR = WORK_DIR / "yolo_dataset"
COURTESY_IMAGES_DIR = WORK_DIR / "courtesy_images"
LEGAL_IMAGES_DIR = WORK_DIR / "legal_images"
TEST_COURTESY_IMAGES_DIR = WORK_DIR / "test_courtesy_images"
TEST_LEGAL_IMAGES_DIR = WORK_DIR / "test_legal_images"

# --- Outputs / checkpoints ---
OUTPUTS_DIR = PROJECT_DIR / "results"
RUNS_DIR = PROJECT_DIR / "runs"
CHECKPOINTS_DIR = PROJECT_DIR / "checkpoints"
RESULTS_FIGURES_DIR = OUTPUTS_DIR / "figures"
PART_B_BEST_CKPT = CHECKPOINTS_DIR / "partB_courtesy_crnn_best.pt"
PART_C_BEST_CKPT = CHECKPOINTS_DIR / "partC_legal_crnn_best.pt"

# --- Validation outputs (existing) ---
PART_A_OUTPUT = OUTPUTS_DIR / "partA_output.txt"
PART_A_METRICS = OUTPUTS_DIR / "partA_metrics.txt"
PART_A_AUDIT = OUTPUTS_DIR / "partA_annotation_audit.json"
PART_A_SPLIT = OUTPUTS_DIR / "partA_split.json"
PART_B_OUTPUT = OUTPUTS_DIR / "partB_output.txt"
PART_B_METRICS = OUTPUTS_DIR / "partB_metrics.txt"
PART_C_OUTPUT = OUTPUTS_DIR / "partC_output.txt"
PART_C_METRICS = OUTPUTS_DIR / "partC_metrics.txt"
PART_D_METRICS = OUTPUTS_DIR / "partD_metrics.txt"

# --- Test-set outputs ---
PART_A_TEST_OUTPUT = OUTPUTS_DIR / "partA_test_output.txt"
PART_A_TEST_METRICS = OUTPUTS_DIR / "partA_test_metrics.txt"
PART_B_TEST_OUTPUT = OUTPUTS_DIR / "partB_test_output.txt"
PART_B_TEST_METRICS = OUTPUTS_DIR / "partB_test_metrics.txt"
PART_C_TEST_OUTPUT = OUTPUTS_DIR / "partC_test_output.txt"
PART_C_TEST_METRICS = OUTPUTS_DIR / "partC_test_metrics.txt"
PART_D_TEST_METRICS = OUTPUTS_DIR / "partD_test_metrics.txt"
FINAL_SUMMARY_TXT = OUTPUTS_DIR / "final_summary.txt"

for directory in [
    WORK_DIR, YOLO_DATASET_DIR, COURTESY_IMAGES_DIR, LEGAL_IMAGES_DIR,
    TEST_COURTESY_IMAGES_DIR, TEST_LEGAL_IMAGES_DIR,
    OUTPUTS_DIR, RUNS_DIR, CHECKPOINTS_DIR, RESULTS_FIGURES_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)

required_dirs = {
    "Images": IMAGES_DIR,
    "BoundingBoxes": BOUNDING_BOXES_DIR,
    "CourtesyAmounts": COURTESY_AMOUNTS_DIR,
    "CourtesyAmounts_raw": COURTESY_AMOUNTS_RAW_DIR,
    "LegalAmounts_raw_text": LEGAL_AMOUNTS_RAW_TEXT_DIR,
    "LegalAmounts": LEGAL_AMOUNTS_TOKENIZED_DIR,
}

missing = [name for name, folder in required_dirs.items() if not folder.exists()]
if missing:
    raise FileNotFoundError(f"Missing required project folders in {PROJECT_DIR}: {missing}")

# Test set is required for evaluation but optional during early development
TEST_AVAILABLE = TEST_IMAGES_DIR.exists() and TEST_BBOX_DIR.exists() \
    and TEST_COURTESY_LABELS.exists() and TEST_LEGAL_LABELS.exists()


def normalize_check_id(filename, prefix=None):
    name = Path(str(filename).lstrip("﻿")).stem
    if prefix and name.startswith(prefix):
        name = name[len(prefix):]
    if name.startswith("ac") and name[2:].isdigit():
        return "ac" + name[2:].zfill(5)
    match = re.search(r"(\d+)", name)
    if match:
        return "ac" + match.group(1).zfill(5)
    return name

print("Project directory:", PROJECT_DIR)
for name, folder in required_dirs.items():
    count = len(list(folder.glob("*.txt"))) if name != "Images" else len(list(folder.glob("*.tif")))
    print(f"{name}: {count} files")
print("Test images:", TEST_IMAGES_DIR, "(available)" if TEST_AVAILABLE else "(MISSING — test eval will be skipped)")
print("Outputs:", OUTPUTS_DIR)
print("Checkpoints:", CHECKPOINTS_DIR)


## Prepare YOLO Dataset

In [ ]:
import json
import math
import random
import shutil
from collections import Counter

IMAGE_EXTENSIONS = {".tif", ".tiff", ".jpg", ".jpeg", ".png"}
YOLO_CLASSES = {0: "legal_amount", 1: "courtesy_amount"}
MIN_BOX_AREA_FOR_REVIEW = 0.001

IMAGES_DIR = Path(IMAGES_DIR)
LABELS_DIR = Path(BOUNDING_BOXES_DIR)
OUTPUT_DIR = str(YOLO_DATASET_DIR)

for split in ["train", "val"]:
    image_split_dir = YOLO_DATASET_DIR / "images" / split
    label_split_dir = YOLO_DATASET_DIR / "labels" / split
    if image_split_dir.exists():
        shutil.rmtree(image_split_dir)
    if label_split_dir.exists():
        shutil.rmtree(label_split_dir)
    image_split_dir.mkdir(parents=True, exist_ok=True)
    label_split_dir.mkdir(parents=True, exist_ok=True)

image_map = {}
duplicate_image_ids = []
for image_path in sorted(IMAGES_DIR.iterdir()):
    if image_path.suffix.lower() not in IMAGE_EXTENSIONS:
        continue
    check_id = normalize_check_id(image_path.name)
    if check_id in image_map:
        duplicate_image_ids.append(check_id)
    image_map[check_id] = image_path

def yolo_to_edges(x_center, y_center, width, height):
    return (
        x_center - width / 2,
        y_center - height / 2,
        x_center + width / 2,
        y_center + height / 2,
    )

def edges_to_yolo(x1, y1, x2, y2):
    width = x2 - x1
    height = y2 - y1
    return ((x1 + x2) / 2, (y1 + y2) / 2, width, height)

def parse_and_clean_label(label_path):
    boxes = []
    warnings = []
    lines = [line.strip() for line in label_path.read_text(encoding="utf-8-sig").splitlines() if line.strip()]

    for line_number, line in enumerate(lines, start=1):
        parts = line.split()
        if len(parts) != 5:
            raise ValueError(f"line {line_number}: expected 5 YOLO fields, found {len(parts)}")

        try:
            class_id = int(parts[0])
            x_center, y_center, width, height = map(float, parts[1:])
        except ValueError as exc:
            raise ValueError(f"line {line_number}: non-numeric YOLO field") from exc

        if class_id not in YOLO_CLASSES:
            raise ValueError(f"line {line_number}: class {class_id} is not one of {sorted(YOLO_CLASSES)}")
        if not all(math.isfinite(value) for value in [x_center, y_center, width, height]):
            raise ValueError(f"line {line_number}: non-finite coordinate")
        if width <= 0 or height <= 0:
            raise ValueError(f"line {line_number}: non-positive box size")
        if not all(0 <= value <= 1 for value in [x_center, y_center, width, height]):
            raise ValueError(f"line {line_number}: YOLO coordinate outside [0, 1]")

        x1, y1, x2, y2 = yolo_to_edges(x_center, y_center, width, height)
        clipped_edges = (max(0.0, x1), max(0.0, y1), min(1.0, x2), min(1.0, y2))
        was_clipped = clipped_edges != (x1, y1, x2, y2)
        x1, y1, x2, y2 = clipped_edges
        if x2 <= x1 or y2 <= y1:
            raise ValueError(f"line {line_number}: box disappears after clipping")

        x_center, y_center, width, height = edges_to_yolo(x1, y1, x2, y2)
        area = width * height
        if was_clipped:
            warnings.append({"file": label_path.name, "line": line_number, "reason": "clipped_to_image_bounds"})
        if area < MIN_BOX_AREA_FOR_REVIEW:
            warnings.append({"file": label_path.name, "line": line_number, "reason": "very_small_box", "area": area})

        boxes.append((class_id, x_center, y_center, width, height))

    class_counts = Counter(class_id for class_id, *_ in boxes)
    if len(boxes) != 2 or class_counts.get(0, 0) != 1 or class_counts.get(1, 0) != 1:
        raise ValueError(f"expected exactly one legal box and one courtesy box, found {dict(class_counts)}")

    return sorted(boxes, key=lambda item: item[0]), warnings

valid_records = []
audit = {
    "duplicate_image_ids": duplicate_image_ids,
    "missing_images_for_labels": [],
    "invalid_label_files": [],
    "warnings": [],
}

for label_path in sorted(LABELS_DIR.glob("*.txt")):
    check_id = normalize_check_id(label_path.name)
    image_path = image_map.get(check_id)
    if image_path is None:
        audit["missing_images_for_labels"].append(label_path.name)
        continue

    try:
        cleaned_boxes, warnings = parse_and_clean_label(label_path)
    except ValueError as exc:
        audit["invalid_label_files"].append({"file": label_path.name, "error": str(exc)})
        continue

    audit["warnings"].extend(warnings)
    valid_records.append({
        "check_id": check_id,
        "image_path": image_path,
        "label_path": label_path,
        "boxes": cleaned_boxes,
    })

if audit["duplicate_image_ids"] or audit["invalid_label_files"]:
    raise ValueError(f"Part-A annotation audit failed. See details: {audit}")
if not valid_records:
    raise ValueError("No valid image-label pairs were found. Check Images/ and BoundingBoxes/ filenames.")

valid_ids = {record["check_id"] for record in valid_records}
audit["images_without_labels"] = sorted(set(image_map) - valid_ids)
audit["valid_image_label_pairs"] = len(valid_records)
audit["total_images"] = len(image_map)
audit["total_label_files"] = len(list(LABELS_DIR.glob("*.txt")))

rng = random.Random(42)
rng.shuffle(valid_records)
split_idx = int(len(valid_records) * 0.85)
train_records = valid_records[:split_idx]
val_records = valid_records[split_idx:]

def write_clean_label(label_path, boxes):
    lines = [f"{class_id} {x:.8f} {y:.8f} {w:.8f} {h:.8f}" for class_id, x, y, w, h in boxes]
    label_path.write_text("\n".join(lines) + "\n", encoding="utf-8")

def copy_records(records, split_name):
    for record in records:
        image_path = record["image_path"]
        target_image = YOLO_DATASET_DIR / "images" / split_name / image_path.name
        target_label = YOLO_DATASET_DIR / "labels" / split_name / f"{image_path.stem}.txt"
        shutil.copy2(image_path, target_image)
        write_clean_label(target_label, record["boxes"])

copy_records(train_records, "train")
copy_records(val_records, "val")

split_summary = {
    "train": [record["check_id"] for record in train_records],
    "val": [record["check_id"] for record in val_records],
    "seed": 42,
    "train_fraction": 0.85,
}
PART_A_AUDIT.write_text(json.dumps(audit, ensure_ascii=False, indent=2), encoding="utf-8")
PART_A_SPLIT.write_text(json.dumps(split_summary, ensure_ascii=False, indent=2), encoding="utf-8")

print("Part-A annotation audit")
print(f"  Images found:              {audit['total_images']}")
print(f"  Label files found:         {audit['total_label_files']}")
print(f"  Valid image-label pairs:   {audit['valid_image_label_pairs']}")
print(f"  Images without labels:     {len(audit['images_without_labels'])}")
print(f"  Non-fatal label warnings:  {len(audit['warnings'])}")
print(f"  Train set:                 {len(train_records)}")
print(f"  Validation set:            {len(val_records)}")
print("YOLO dataset prepared at:", YOLO_DATASET_DIR)
print("Audit saved to:", PART_A_AUDIT)
print("Split saved to:", PART_A_SPLIT)


## Convert TIF to JPG

In [ ]:
from PIL import Image


def convert_split_to_jpg(split_name):
    image_dir = YOLO_DATASET_DIR / "images" / split_name
    tif_paths = sorted(list(image_dir.glob("*.tif")) + list(image_dir.glob("*.tiff")))
    converted = 0

    for tif_path in tif_paths:
        jpg_path = tif_path.with_suffix(".jpg")
        with Image.open(tif_path) as image:
            image.convert("RGB").save(jpg_path, "JPEG", quality=95)
        tif_path.unlink()
        converted += 1

    jpg_count = len(list(image_dir.glob("*.jpg")))
    print(f"{split_name}: converted {converted} TIFF files; {jpg_count} JPG files ready")

print("Converting YOLO images to JPG")
convert_split_to_jpg("train")
convert_split_to_jpg("val")


## Train YOLO Model

In [ ]:
import yaml
from ultralytics import YOLO

data_yaml = {
    "path": str(YOLO_DATASET_DIR),
    "train": "images/train",
    "val": "images/val",
    "nc": 2,
    "names": {0: "legal_amount", 1: "courtesy_amount"},
}

YAML_PATH = YOLO_DATASET_DIR / "data.yaml"
YAML_PATH.write_text(yaml.dump(data_yaml, default_flow_style=False), encoding="utf-8")

print("data.yaml created at:", YAML_PATH)
print(yaml.dump(data_yaml, default_flow_style=False))

model = YOLO("yolov8n.pt")

results = model.train(
    data=str(YAML_PATH),
    epochs=100,
    batch=16,
    imgsz=640,
    optimizer="AdamW",
    lr0=0.001,
    momentum=0.9,
    weight_decay=0.0005,
    amp=True,
    patience=100,
    project=str(RUNS_DIR),
    name="partA_yolov8n",
    exist_ok=True,
    save=True,
    verbose=True,
    deterministic=False,
)

print("Training complete")


## Part A - Generate Predictions

In [ ]:
import glob
from PIL import Image
from ultralytics import YOLO


def latest_best_model():
    candidates = []
    candidates.extend(glob.glob(str(RUNS_DIR / "**" / "weights" / "best.pt"), recursive=True))
    candidates.extend(glob.glob("/content/runs/detect/train*/weights/best.pt"))
    candidates.extend(glob.glob("/content/runs/train*/weights/best.pt"))
    if not candidates:
        raise FileNotFoundError("No best.pt file was found. Run the Part-A YOLO training cell first.")
    return max(candidates, key=os.path.getmtime)


def clip_xyxy(box, width, height):
    x1, y1, x2, y2 = [int(round(value)) for value in box]
    x1 = max(0, min(width, x1))
    y1 = max(0, min(height, y1))
    x2 = max(0, min(width, x2))
    y2 = max(0, min(height, y2))
    if x2 <= x1 or y2 <= y1:
        return [0, 0, 0, 0]
    return [x1, y1, x2, y2]

BEST_MODEL_PATH = latest_best_model()
model = YOLO(BEST_MODEL_PATH)
print("Using model:", BEST_MODEL_PATH)

VAL_IMAGES_DIR = YOLO_DATASET_DIR / "images" / "val"
val_images = sorted(VAL_IMAGES_DIR.glob("*.jpg"))
if not val_images:
    raise ValueError(f"No validation JPG images found in {VAL_IMAGES_DIR}")

output_lines = []
missed = {"courtesy_amount": 0, "legal_amount": 0}

for image_path in val_images:
    with Image.open(image_path) as image:
        image_width, image_height = image.size

    result = model.predict(str(image_path), imgsz=640, conf=0.25, verbose=False)[0]
    best_by_class = {
        0: {"confidence": -1.0, "box": [0, 0, 0, 0]},
        1: {"confidence": -1.0, "box": [0, 0, 0, 0]},
    }

    for box in result.boxes:
        class_id = int(box.cls[0].item())
        if class_id not in best_by_class:
            continue
        confidence = float(box.conf[0].item())
        if confidence <= best_by_class[class_id]["confidence"]:
            continue
        best_by_class[class_id] = {
            "confidence": confidence,
            "box": clip_xyxy(box.xyxy[0].tolist(), image_width, image_height),
        }

    legal_box = best_by_class[0]["box"]
    courtesy_box = best_by_class[1]["box"]
    if courtesy_box == [0, 0, 0, 0]:
        missed["courtesy_amount"] += 1
    if legal_box == [0, 0, 0, 0]:
        missed["legal_amount"] += 1

    output_name = image_path.with_suffix(".tif").name
    output_lines.append(
        f"{output_name} "
        f"{courtesy_box[0]} {courtesy_box[1]} {courtesy_box[2]} {courtesy_box[3]} "
        f"{legal_box[0]} {legal_box[1]} {legal_box[2]} {legal_box[3]}"
    )

PART_A_OUTPUT.write_text("\n".join(output_lines) + "\n", encoding="utf-8")

print("PART A OUTPUT SUMMARY")
print(f"  Validation images processed: {len(output_lines)}")
print(f"  Missed courtesy fields:      {missed['courtesy_amount']}")
print(f"  Missed legal fields:         {missed['legal_amount']}")
print(f"  Output saved to:             {PART_A_OUTPUT}")
print("Sample output:")
for line in output_lines[:5]:
    print(" ", line)


## Helper Functions for Evaluation

In [ ]:
import numpy as np
from PIL import Image

IOU_THRESHOLDS = (0.50, 0.75, 0.90)
CLASS_NAMES = {0: "legal_amount", 1: "courtesy_amount"}


def calculate_iou(box1, box2):
    x_left = max(box1[0], box2[0])
    y_top = max(box1[1], box2[1])
    x_right = min(box1[2], box2[2])
    y_bottom = min(box1[3], box2[3])

    intersection = max(0, x_right - x_left) * max(0, y_bottom - y_top)
    area1 = max(0, box1[2] - box1[0]) * max(0, box1[3] - box1[1])
    area2 = max(0, box2[2] - box2[0]) * max(0, box2[3] - box2[1])
    union = area1 + area2 - intersection
    return 0.0 if union <= 0 else intersection / union


def convert_yolo_to_abs_coords(x_center, y_center, width, height, img_width, img_height):
    x1 = max(0, int(round((x_center - width / 2) * img_width)))
    y1 = max(0, int(round((y_center - height / 2) * img_height)))
    x2 = min(img_width, int(round((x_center + width / 2) * img_width)))
    y2 = min(img_height, int(round((y_center + height / 2) * img_height)))
    return [x1, y1, x2, y2]


def summarize_iou_scores(iou_scores, thresholds=IOU_THRESHOLDS):
    total = len(iou_scores)
    mean_iou = float(np.mean(iou_scores)) if iou_scores else 0.0
    accuracies = {threshold: 0.0 for threshold in thresholds}
    if total:
        accuracies = {threshold: sum(iou >= threshold for iou in iou_scores) / total for threshold in thresholds}
    return {"total": total, "mean_iou": mean_iou, "accuracies": accuracies}


## Load Ground Truth and Predictions

In [ ]:
VAL_IMAGES_DIR = YOLO_DATASET_DIR / "images" / "val"
VAL_LABELS_DIR = YOLO_DATASET_DIR / "labels" / "val"
val_images = sorted(VAL_IMAGES_DIR.glob("*.jpg"))

if not val_images:
    raise ValueError(f"No validation images found in {VAL_IMAGES_DIR}")
if not PART_A_OUTPUT.exists():
    raise FileNotFoundError(f"Part-A prediction file not found: {PART_A_OUTPUT}")

image_dimensions = {}
ground_truths = {}

for image_path in val_images:
    with Image.open(image_path) as image:
        image_dimensions[image_path.stem] = image.size

    label_path = VAL_LABELS_DIR / f"{image_path.stem}.txt"
    if not label_path.exists():
        raise FileNotFoundError(f"Missing validation label file: {label_path}")

    boxes = {}
    image_width, image_height = image_dimensions[image_path.stem]
    for line_number, line in enumerate(label_path.read_text(encoding="utf-8").splitlines(), start=1):
        if not line.strip():
            continue
        parts = line.split()
        if len(parts) != 5:
            raise ValueError(f"Malformed label line in {label_path}, line {line_number}: {line}")
        class_id = int(parts[0])
        x_center, y_center, width, height = map(float, parts[1:])
        boxes[class_id] = convert_yolo_to_abs_coords(x_center, y_center, width, height, image_width, image_height)

    if set(boxes) != {0, 1}:
        raise ValueError(f"Expected one legal and one courtesy box in {label_path}, found classes {sorted(boxes)}")
    ground_truths[image_path.stem] = boxes

predictions = {}
for line_number, line in enumerate(PART_A_OUTPUT.read_text(encoding="utf-8").splitlines(), start=1):
    if not line.strip():
        continue
    parts = line.split()
    if len(parts) != 9:
        raise ValueError(f"Malformed Part-A output line {line_number}: {line}")
    check_id = Path(parts[0]).stem
    values = [int(value) for value in parts[1:]]
    predictions[check_id] = {
        1: values[0:4],
        0: values[4:8],
    }

missing_predictions = sorted(set(ground_truths) - set(predictions))
extra_predictions = sorted(set(predictions) - set(ground_truths))
if missing_predictions or extra_predictions:
    raise ValueError(
        f"Prediction/validation mismatch. Missing={missing_predictions[:10]}, extra={extra_predictions[:10]}"
    )

print(f"Loaded {len(ground_truths)} validation ground-truth samples")
print(f"Loaded {len(predictions)} Part-A prediction rows")


## Calculate Part A Metrics

In [ ]:
per_class_ious = {0: [], 1: []}
missed_predictions = {0: 0, 1: 0}

for check_id in sorted(ground_truths):
    for class_id in [0, 1]:
        gt_box = ground_truths[check_id][class_id]
        pred_box = predictions[check_id][class_id]
        if pred_box == [0, 0, 0, 0]:
            missed_predictions[class_id] += 1
            per_class_ious[class_id].append(0.0)
        else:
            per_class_ious[class_id].append(calculate_iou(pred_box, gt_box))

metrics = {class_id: summarize_iou_scores(scores) for class_id, scores in per_class_ious.items()}
combined_scores = per_class_ious[0] + per_class_ious[1]
metrics["overall"] = summarize_iou_scores(combined_scores)

report_lines = []
report_lines.append("PART A - LEGAL AND COURTESY AMOUNT EXTRACTION METRICS")
report_lines.append(f"Validation images: {len(ground_truths)}")
report_lines.append(f"Prediction file: {PART_A_OUTPUT}")
report_lines.append("")

for class_id in [1, 0]:
    class_name = CLASS_NAMES[class_id]
    class_metrics = metrics[class_id]
    report_lines.append(class_name)
    report_lines.append(f"  Total samples: {class_metrics['total']}")
    report_lines.append(f"  Missed predictions: {missed_predictions[class_id]}")
    report_lines.append(f"  Mean IoU: {class_metrics['mean_iou']:.4f}")
    for threshold, accuracy in class_metrics["accuracies"].items():
        report_lines.append(f"  Accuracy @ IoU {threshold:.2f}: {accuracy:.4f}")
    report_lines.append("")

overall_metrics = metrics["overall"]
report_lines.append("overall")
report_lines.append(f"  Total boxes: {overall_metrics['total']}")
report_lines.append(f"  Mean IoU: {overall_metrics['mean_iou']:.4f}")
for threshold, accuracy in overall_metrics["accuracies"].items():
    report_lines.append(f"  Accuracy @ IoU {threshold:.2f}: {accuracy:.4f}")

PART_A_METRICS.write_text("\n".join(report_lines) + "\n", encoding="utf-8")
print("\n".join(report_lines))
print("\nMetrics saved to:", PART_A_METRICS)


## Part A — Test Set Evaluation

In [ ]:
# === Part A — Test set evaluation (CENPARMI 600-image test split) ===
import glob
from PIL import Image
from ultralytics import YOLO

if not TEST_AVAILABLE:
    print("Test set not available — skipping Part A test evaluation.")
else:
    def latest_best_model():
        candidates = []
        candidates.extend(glob.glob(str(RUNS_DIR / "**" / "weights" / "best.pt"), recursive=True))
        candidates.extend(glob.glob("/content/runs/detect/train*/weights/best.pt"))
        candidates.extend(glob.glob("/content/runs/train*/weights/best.pt"))
        if not candidates:
            raise FileNotFoundError("No best.pt found. Run Part A training first.")
        return max(candidates, key=os.path.getmtime)

    BEST_MODEL_PATH = latest_best_model()
    yolo_test = YOLO(BEST_MODEL_PATH)
    print("Using YOLO model:", BEST_MODEL_PATH)

    test_images = sorted(TEST_IMAGES_DIR.glob("*.tif"))
    print(f"Found {len(test_images)} test images.")

    def clip_xyxy(box, w, h):
        x1, y1, x2, y2 = [int(round(v)) for v in box]
        x1 = max(0, min(w, x1)); y1 = max(0, min(h, y1))
        x2 = max(0, min(w, x2)); y2 = max(0, min(h, y2))
        if x2 <= x1 or y2 <= y1:
            return [0, 0, 0, 0]
        return [x1, y1, x2, y2]

    test_predictions = {}
    test_dimensions = {}
    test_output_lines = []
    missed = {"courtesy_amount": 0, "legal_amount": 0}

    for image_path in test_images:
        with Image.open(image_path) as img:
            iw, ih = img.size
        test_dimensions[image_path.stem] = (iw, ih)

        result = yolo_test.predict(str(image_path), imgsz=640, conf=0.25, verbose=False)[0]
        best = {0: {"conf": -1, "box": [0, 0, 0, 0]}, 1: {"conf": -1, "box": [0, 0, 0, 0]}}
        for box in result.boxes:
            cid = int(box.cls[0].item())
            if cid not in best:
                continue
            cf = float(box.conf[0].item())
            if cf <= best[cid]["conf"]:
                continue
            best[cid] = {"conf": cf, "box": clip_xyxy(box.xyxy[0].tolist(), iw, ih)}

        legal_box = best[0]["box"]
        courtesy_box = best[1]["box"]
        if courtesy_box == [0, 0, 0, 0]:
            missed["courtesy_amount"] += 1
        if legal_box == [0, 0, 0, 0]:
            missed["legal_amount"] += 1

        test_predictions[image_path.stem] = {1: courtesy_box, 0: legal_box}
        test_output_lines.append(
            f"{image_path.name} "
            f"{courtesy_box[0]} {courtesy_box[1]} {courtesy_box[2]} {courtesy_box[3]} "
            f"{legal_box[0]} {legal_box[1]} {legal_box[2]} {legal_box[3]}"
        )

    PART_A_TEST_OUTPUT.write_text("\n".join(test_output_lines) + "\n", encoding="utf-8")

    # ----- IoU vs ground truth -----
    test_gt = {}
    for label_path in sorted(TEST_BBOX_DIR.glob("*.txt")):
        cid = label_path.stem
        if cid not in test_dimensions:
            continue
        iw, ih = test_dimensions[cid]
        boxes = {}
        for line in label_path.read_text(encoding="utf-8").splitlines():
            line = line.strip()
            if not line:
                continue
            parts = line.split()
            if len(parts) != 5:
                continue
            class_id = int(parts[0])
            xc, yc, w, h = map(float, parts[1:])
            boxes[class_id] = convert_yolo_to_abs_coords(xc, yc, w, h, iw, ih)
        if 0 in boxes and 1 in boxes:
            test_gt[cid] = boxes

    per_class_ious = {0: [], 1: []}
    missed_pred = {0: 0, 1: 0}
    for cid, gt in test_gt.items():
        pred = test_predictions.get(cid)
        if pred is None:
            for k in [0, 1]:
                missed_pred[k] += 1
                per_class_ious[k].append(0.0)
            continue
        for k in [0, 1]:
            pb = pred[k]
            if pb == [0, 0, 0, 0]:
                missed_pred[k] += 1
                per_class_ious[k].append(0.0)
            else:
                per_class_ious[k].append(calculate_iou(pb, gt[k]))

    metrics_test = {k: summarize_iou_scores(v) for k, v in per_class_ious.items()}
    metrics_test["overall"] = summarize_iou_scores(per_class_ious[0] + per_class_ious[1])

    lines = ["PART A — TEST SET METRICS",
             f"Test images evaluated: {len(test_gt)}",
             f"Prediction file: {PART_A_TEST_OUTPUT}", ""]
    for cid in [1, 0]:
        m = metrics_test[cid]
        lines.append(CLASS_NAMES[cid])
        lines.append(f"  Total samples: {m['total']}")
        lines.append(f"  Missed predictions: {missed_pred[cid]}")
        lines.append(f"  Mean IoU: {m['mean_iou']:.4f}")
        for t, a in m["accuracies"].items():
            lines.append(f"  Accuracy @ IoU {t:.2f}: {a:.4f}")
        lines.append("")
    om = metrics_test["overall"]
    lines.append("overall")
    lines.append(f"  Total boxes: {om['total']}")
    lines.append(f"  Mean IoU: {om['mean_iou']:.4f}")
    for t, a in om["accuracies"].items():
        lines.append(f"  Accuracy @ IoU {t:.2f}: {a:.4f}")

    PART_A_TEST_METRICS.write_text("\n".join(lines) + "\n", encoding="utf-8")
    print("\n".join(lines))
    print(f"\nSaved to: {PART_A_TEST_METRICS}")


## Part B — Data Organization and Cropping

In [ ]:
import os
import cv2
import glob
import zipfile
from ultralytics import YOLO

# 1. Define all paths clearly
TEXT_DIR = str(COURTESY_AMOUNTS_DIR)
OUTPUT_IMAGES_DIR = str(COURTESY_IMAGES_DIR)
RAW_IMAGES_DIR = str(IMAGES_DIR)

print("--- Step 1: Data Organization & Auto-Cropping ---")

# 2. Courtesy token files already exist in the Google Drive project folder.
os.makedirs(TEXT_DIR, exist_ok=True)
print(f"Using courtesy labels from: {TEXT_DIR}")

# 3. Create the new dedicated images folder
os.makedirs(OUTPUT_IMAGES_DIR, exist_ok=True)
print(f"Created new directory for crops: {OUTPUT_IMAGES_DIR}")

# 4. Auto-find the best.pt weights (using your robust glob method)
all_best_models = glob.glob(str(RUNS_DIR / "**" / "weights" / "best.pt"), recursive=True) + \
                  glob.glob("/content/runs/detect/train*/weights/best.pt") + \
                  glob.glob("/content/runs/train*/weights/best.pt")

if not all_best_models:
    raise FileNotFoundError("No best.pt found! Please re-run the YOLO training.")

latest_weights = max(all_best_models, key=os.path.getmtime)
model = YOLO(latest_weights)
print(f"Loaded YOLO model: {latest_weights}")

# 5. Process raw images and save crops
raw_images = glob.glob(os.path.join(RAW_IMAGES_DIR, '*.tif'))
print(f"Found {len(raw_images)} raw images. Cropping Courtesy boxes...")

success_count = 0

for img_path in raw_images:
    base_name = os.path.basename(img_path)
    expected_crop_name = f"C{base_name}"   # e.g., Cac00000.tif

    # Read with OpenCV to enforce 3-channel format and avoid the grayscale error
    img = cv2.imread(img_path)
    if img is None:
        continue

    results = model.predict(img, conf=0.25, verbose=False)

    if not results or len(results[0].boxes) == 0:
        continue

    # Find the Courtesy Amount (Class 1)
    for box in results[0].boxes:
        cls_id = int(box.cls[0].item())
        if cls_id == 1:
            # Get coordinates and crop
            x1, y1, x2, y2 = [int(v) for v in box.xyxy[0].tolist()]
            x1, y1 = max(0, x1), max(0, y1)
            x2, y2 = min(img.shape[1], x2), min(img.shape[0], y2)

            crop_img = img[y1:y2, x1:x2]

            # Save to the NEW directory
            save_path = os.path.join(OUTPUT_IMAGES_DIR, expected_crop_name)
            cv2.imwrite(save_path, crop_img)
            success_count += 1
            break

print(f"\n✅ Success! Generated {success_count} cropped images in {OUTPUT_IMAGES_DIR}.")


## Verify Cropped Images

In [ ]:
image_count = len(list(COURTESY_IMAGES_DIR.glob("*.tif"))) + len(list(COURTESY_IMAGES_DIR.glob("*.jpg")))
label_count = sum(1 for _ in COURTESY_AMOUNTS_DIR.glob("*.txt"))
print("Number of courtesy crop images found:", image_count)
print("Number of courtesy TXT label files found:", label_count)
print("Courtesy labels directory:", COURTESY_AMOUNTS_DIR)


## Part B — CRNN Dataloader

In [ ]:
import os
import ast
import glob
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import editdistance

In [ ]:
class CourtesyDataset(Dataset):
    def __init__(self, text_dir, img_dir, transform=None):
        self.transform = transform
        self.data = []
        self.blank_idx = 10

        txt_files = glob.glob(os.path.join(text_dir, '**/*.txt'), recursive=True)

        image_map = {}
        for root, _, files in os.walk(img_dir):
            for f in files:
                if f.lower().endswith(('.tif', '.tiff', '.jpg', '.jpeg', '.png')):
                    key = normalize_check_id(f, prefix='C')
                    image_map[key] = os.path.join(root, f)

        for txt_file in txt_files:
            with open(txt_file, 'r', encoding='utf-8-sig') as f:
                for line in f:
                    parts = line.strip().split('\t')
                    if len(parts) != 2:
                        parts = line.strip().split(' ', 1)
                    if len(parts) != 2:
                        continue

                    img_name, seq_str = parts[0].strip(), parts[1].strip()
                    img_key = normalize_check_id(img_name, prefix='C')
                    img_path = image_map.get(img_key)
                    if not img_path:
                        continue

                    seq_str = seq_str.replace('\u202a', '').replace('\u202c', '')
                    tokens = re.findall(r'\d+|[./]', seq_str)
                    clean_seq = [int(token) for token in tokens if token.isdigit() and token != str(self.blank_idx)]

                    if clean_seq:
                        self.data.append((img_path, clean_seq))

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        img_path, seq = self.data[idx]
        img = Image.open(img_path).convert('L')
        if self.transform:
            img = self.transform(img)
        return img, torch.tensor(seq, dtype=torch.long), os.path.basename(img_path)

def collate_fn(batch):
    images, targets, filenames = zip(*batch)
    images = torch.stack(images)
    target_lengths = torch.tensor([len(t) for t in targets], dtype=torch.long)
    targets = torch.nn.utils.rnn.pad_sequence(targets, batch_first=True, padding_value=0)
    return images, targets, target_lengths, filenames

transform = transforms.Compose([
    transforms.Resize((32, 128)),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

dataset = CourtesyDataset(text_dir=str(COURTESY_AMOUNTS_DIR), img_dir=str(COURTESY_IMAGES_DIR), transform=transform)
print(f"Loaded {len(dataset)} valid image-sequence pairs.")

if len(dataset) == 0:
    raise ValueError("Dataset is empty. Ensure courtesy cropping succeeded and labels are in CourtesyAmounts/.")

train_size = int(0.85 * len(dataset))
val_size = len(dataset) - train_size
train_dataset, val_dataset = torch.utils.data.random_split(dataset, [train_size, val_size], generator=torch.Generator().manual_seed(42))

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(val_dataset, batch_size=1, shuffle=False, collate_fn=collate_fn)


## Part B — CRNN Model Definition

In [ ]:
class CRNN(nn.Module):
    def __init__(self, num_classes):
        super(CRNN, self).__init__()
        self.cnn = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2, 2),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2, 2),
            nn.Conv2d(64, 128, 3, padding=1), nn.ReLU(), nn.MaxPool2d((2, 2), (2, 1))
        )
        self.pool = nn.AdaptiveAvgPool2d((1, None))
        self.rnn = nn.LSTM(128, 64, bidirectional=True, batch_first=True)
        self.fc = nn.Linear(128, num_classes)

    def forward(self, x):
        x = self.cnn(x)
        x = self.pool(x).squeeze(2).permute(0, 2, 1)
        x, _ = self.rnn(x)
        x = self.fc(x)
        return x.log_softmax(2).permute(1, 0, 2)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
crnn_model = CRNN(num_classes=11).to(device)
print(f"CRNN Model initialized on {device.type.upper()}.")


## Part B — CRNN Training

In [ ]:
# === Part B — Training (RMSprop @ 1e-4, patience 20, best ckpt) ===
import copy

PART_B_EPOCHS = 500
PART_B_PATIENCE = 20
PART_B_LR = 1e-4

criterion = nn.CTCLoss(blank=10, zero_infinity=True)
optimizer = optim.RMSprop(crnn_model.parameters(), lr=PART_B_LR)

best_val_loss = float("inf")
best_state = None
epochs_since_improve = 0
history = []

print(f"Training Part B for up to {PART_B_EPOCHS} epochs (patience {PART_B_PATIENCE})...")
for epoch in range(PART_B_EPOCHS):
    crnn_model.train()
    train_loss_sum = 0.0
    for images, targets, target_lengths, _ in train_loader:
        images, targets = images.to(device), targets.to(device)
        optimizer.zero_grad()
        outputs = crnn_model(images)
        input_lengths = torch.full((images.size(0),), outputs.size(0), dtype=torch.long)
        loss = criterion(outputs, targets, input_lengths, target_lengths)
        loss.backward()
        optimizer.step()
        train_loss_sum += loss.item()
    train_loss = train_loss_sum / max(1, len(train_loader))

    crnn_model.eval()
    val_loss_sum = 0.0
    n_val_batches = 0
    with torch.no_grad():
        for images, targets, target_lengths, _ in val_loader:
            images, targets = images.to(device), targets.to(device)
            outputs = crnn_model(images)
            input_lengths = torch.full((images.size(0),), outputs.size(0), dtype=torch.long)
            v = criterion(outputs, targets, input_lengths, target_lengths)
            val_loss_sum += v.item()
            n_val_batches += 1
    val_loss = val_loss_sum / max(1, n_val_batches)
    history.append({"epoch": epoch + 1, "train_loss": train_loss, "val_loss": val_loss})

    improved = val_loss < best_val_loss - 1e-6
    if improved:
        best_val_loss = val_loss
        best_state = copy.deepcopy(crnn_model.state_dict())
        torch.save(best_state, PART_B_BEST_CKPT)
        epochs_since_improve = 0
    else:
        epochs_since_improve += 1

    if (epoch + 1) % 5 == 0 or epoch == 0 or improved:
        print(f"Epoch {epoch+1:03d} | train {train_loss:.4f} | val {val_loss:.4f}"
              f" | best {best_val_loss:.4f} | wait {epochs_since_improve}")

    if epochs_since_improve >= PART_B_PATIENCE:
        print(f"Early stopping at epoch {epoch+1} (no improvement for {PART_B_PATIENCE} epochs).")
        break

if best_state is not None:
    crnn_model.load_state_dict(best_state)
print(f"Best val loss: {best_val_loss:.4f}")
print(f"Best checkpoint saved to: {PART_B_BEST_CKPT}")


## Part B — CRNN Evaluation

In [ ]:
# === Part B — Validation evaluation (digit + amount-exact + 0/1/2+) ===
crnn_model.eval()

def decode_b_preds(preds, blank_idx=10):
    decoded, prev = [], -1
    for c in preds:
        v = c.item() if hasattr(c, "item") else int(c)
        if v != prev and v != blank_idx:
            decoded.append(str(v))
        prev = v
    return "".join(decoded)

total_N, total_errors = 0, 0
exact_match = 0
error_buckets = {0: 0, 1: 0, "2+": 0}
output_lines = []
b_pred_strings = {}
b_gt_strings = {}

with torch.no_grad():
    for images, targets, target_lengths, filenames in val_loader:
        images = images.to(device)
        outputs = crnn_model(images)
        _, preds = outputs.max(2)
        pred_str = decode_b_preds(preds.transpose(1, 0)[0])
        gt_str = "".join(str(c.item()) for c in targets[0][:target_lengths[0]])
        output_lines.append(f"{filenames[0]} {pred_str}")
        b_pred_strings[filenames[0]] = pred_str
        b_gt_strings[filenames[0]] = gt_str

        N = len(gt_str)
        if N == 0:
            continue
        errs = editdistance.eval(pred_str, gt_str)
        total_N += N
        total_errors += errs
        if errs == 0:
            error_buckets[0] += 1
            exact_match += 1
        elif errs == 1:
            error_buckets[1] += 1
        else:
            error_buckets["2+"] += 1

total_samples = sum(error_buckets.values())
digit_acc = max(0.0, (1 - total_errors / total_N) * 100) if total_N else 0.0
amount_acc = (exact_match / total_samples) * 100 if total_samples else 0.0

lines = ["PART B — COURTESY AMOUNT (VALIDATION)",
         f"Samples: {total_samples}",
         f"Accuracy at digit level: {digit_acc:.2f}%",
         f"Amount-level (exact) accuracy: {amount_acc:.2f}% ({exact_match}/{total_samples})",
         f"Amounts with no errors: {error_buckets[0]/total_samples*100:.2f}% ({error_buckets[0]})",
         f"Amounts with 1 error:   {error_buckets[1]/total_samples*100:.2f}% ({error_buckets[1]})",
         f"Amounts with 2+ errors: {error_buckets['2+']/total_samples*100:.2f}% ({error_buckets['2+']})"]
PART_B_OUTPUT.write_text("\n".join(output_lines) + "\n", encoding="utf-8")
PART_B_METRICS.write_text("\n".join(lines) + "\n", encoding="utf-8")
print("\n".join(lines))
print("Output saved to:", PART_B_OUTPUT)


## Part B — Confusion Matrix (Validation)

In [ ]:
# === Part B — Confusion matrix (validation) ===
import numpy as np
import matplotlib.pyplot as plt

def build_digit_confusion(pred_strings, gt_strings, blank=10):
    # 10 digits + Insertion row + Deletion column
    cm = np.zeros((11, 11), dtype=int)
    for fname, gt in gt_strings.items():
        pred = pred_strings.get(fname, "")
        # Levenshtein-style alignment via DP
        n, m = len(gt), len(pred)
        dp = np.zeros((n + 1, m + 1), dtype=int)
        for i in range(n + 1):
            dp[i, 0] = i
        for j in range(m + 1):
            dp[0, j] = j
        for i in range(1, n + 1):
            for j in range(1, m + 1):
                cost = 0 if gt[i - 1] == pred[j - 1] else 1
                dp[i, j] = min(dp[i - 1, j] + 1, dp[i, j - 1] + 1, dp[i - 1, j - 1] + cost)
        # backtrack
        i, j = n, m
        while i > 0 or j > 0:
            if i > 0 and j > 0 and dp[i, j] == dp[i - 1, j - 1] + (0 if gt[i - 1] == pred[j - 1] else 1):
                gd = int(gt[i - 1]); pd = int(pred[j - 1])
                cm[gd, pd] += 1
                i -= 1; j -= 1
            elif i > 0 and dp[i, j] == dp[i - 1, j] + 1:
                gd = int(gt[i - 1])
                cm[gd, 10] += 1  # deletion
                i -= 1
            else:
                pd = int(pred[j - 1])
                cm[10, pd] += 1  # insertion
                j -= 1
    return cm

def plot_digit_cm(cm, title, save_path):
    fig, ax = plt.subplots(figsize=(8, 7))
    ax.imshow(cm, cmap="Blues")
    ax.set_xticks(range(11)); ax.set_yticks(range(11))
    ax.set_xticklabels([str(i) for i in range(10)] + ["Deletion"], rotation=45)
    ax.set_yticklabels([str(i) for i in range(10)] + ["Insertion"])
    ax.set_xlabel("Predicted Labels"); ax.set_ylabel("True Labels")
    ax.set_title(title)
    for i in range(11):
        for j in range(11):
            v = cm[i, j]
            if v > 0:
                ax.text(j, i, str(v), ha="center", va="center",
                        color="white" if v > cm.max() * 0.5 else "black", fontsize=8)
    plt.tight_layout()
    plt.savefig(save_path, dpi=120)
    plt.show()
    print("Saved:", save_path)

cm_b_val = build_digit_confusion(b_pred_strings, b_gt_strings)
plot_digit_cm(cm_b_val, "Part B Confusion Matrix (Validation)",
              RESULTS_FIGURES_DIR / "partB_confusion_val.png")


## Part B — Test Set Evaluation

In [ ]:
# === Part B — Test set evaluation ===
import cv2
import shutil
from torch.utils.data import Dataset, DataLoader

if not TEST_AVAILABLE:
    print("Test set not available — skipping Part B test evaluation.")
else:
    # 1. Crop test courtesy regions with YOLO
    yolo_for_crop = YOLO(BEST_MODEL_PATH) if 'yolo_for_crop' not in dir() else yolo_for_crop
    test_images = sorted(TEST_IMAGES_DIR.glob("*.tif"))
    crop_count = 0
    for img_path in test_images:
        out_name = f"C{img_path.name}"
        out_path = TEST_COURTESY_IMAGES_DIR / out_name
        if out_path.exists():
            crop_count += 1
            continue
        img = cv2.imread(str(img_path))
        if img is None:
            continue
        result = yolo_for_crop.predict(img, conf=0.25, verbose=False)
        if not result or len(result[0].boxes) == 0:
            continue
        for box in result[0].boxes:
            if int(box.cls[0].item()) == 1:
                x1, y1, x2, y2 = [int(v) for v in box.xyxy[0].tolist()]
                x1, y1 = max(0, x1), max(0, y1)
                x2, y2 = min(img.shape[1], x2), min(img.shape[0], y2)
                cv2.imwrite(str(out_path), img[y1:y2, x1:x2])
                crop_count += 1
                break
    print(f"Test courtesy crops ready: {crop_count}/{len(test_images)} in {TEST_COURTESY_IMAGES_DIR}")

    # 2. Stage just the courtesy labels into a clean dir to keep the
    #    recursive .txt scan from also reading BoundingBox/ and LegalAmounts.txt.
    test_courtesy_label_dir = WORK_DIR / "test_courtesy_labels"
    test_courtesy_label_dir.mkdir(parents=True, exist_ok=True)
    shutil.copy2(TEST_COURTESY_LABELS, test_courtesy_label_dir / "CourtesyAmounts.txt")

    test_dataset_b = CourtesyDataset(
        text_dir=str(test_courtesy_label_dir),
        img_dir=str(TEST_COURTESY_IMAGES_DIR),
        transform=transform,
    )
    print(f"Test pairs (Part B): {len(test_dataset_b)}")
    test_loader_b = DataLoader(test_dataset_b, batch_size=1, shuffle=False, collate_fn=collate_fn)

    # 3. Eval (digit + amount-exact + 0/1/2+ + confusion matrix)
    crnn_model.load_state_dict(torch.load(PART_B_BEST_CKPT, map_location=device))
    crnn_model.eval()

    total_N, total_errors = 0, 0
    exact_match = 0
    err_buckets = {0: 0, 1: 0, "2+": 0}
    out_lines = []
    pred_strs_test = {}; gt_strs_test = {}
    with torch.no_grad():
        for images, targets, target_lengths, filenames in test_loader_b:
            images = images.to(device)
            outputs = crnn_model(images)
            _, preds = outputs.max(2)
            pred_str = decode_b_preds(preds.transpose(1, 0)[0])
            gt_str = "".join(str(c.item()) for c in targets[0][:target_lengths[0]])
            out_lines.append(f"{filenames[0]} {pred_str}")
            pred_strs_test[filenames[0]] = pred_str
            gt_strs_test[filenames[0]] = gt_str

            N = len(gt_str)
            if N == 0:
                continue
            errs = editdistance.eval(pred_str, gt_str)
            total_N += N; total_errors += errs
            if errs == 0:
                err_buckets[0] += 1; exact_match += 1
            elif errs == 1:
                err_buckets[1] += 1
            else:
                err_buckets["2+"] += 1

    total = sum(err_buckets.values())
    digit_acc = max(0.0, (1 - total_errors / total_N) * 100) if total_N else 0.0
    amount_acc = (exact_match / total) * 100 if total else 0.0

    lines = ["PART B — COURTESY AMOUNT (TEST)",
             f"Samples: {total}",
             f"Accuracy at digit level: {digit_acc:.2f}%",
             f"Amount-level (exact) accuracy: {amount_acc:.2f}% ({exact_match}/{total})",
             f"Amounts with no errors: {err_buckets[0]/total*100:.2f}% ({err_buckets[0]})",
             f"Amounts with 1 error:   {err_buckets[1]/total*100:.2f}% ({err_buckets[1]})",
             f"Amounts with 2+ errors: {err_buckets['2+']/total*100:.2f}% ({err_buckets['2+']})"]
    PART_B_TEST_OUTPUT.write_text("\n".join(out_lines) + "\n", encoding="utf-8")
    PART_B_TEST_METRICS.write_text("\n".join(lines) + "\n", encoding="utf-8")
    print("\n".join(lines))

    cm_b_test = build_digit_confusion(pred_strs_test, gt_strs_test)
    plot_digit_cm(cm_b_test, "Part B Confusion Matrix (Test)",
                  RESULTS_FIGURES_DIR / "partB_confusion_test.png")


## Part C — Legal Amount Cropping

In [ ]:
import os
import cv2
import glob
import zipfile
from ultralytics import YOLO

# 1. Define Paths
TEXT_DIR = str(LEGAL_AMOUNTS_TOKENIZED_DIR)
OUTPUT_IMAGES_DIR = str(LEGAL_IMAGES_DIR)
RAW_IMAGES_DIR = str(IMAGES_DIR)

print("--- Step 1: Legal Amount Data Extraction & Auto-Cropping ---")

# 2. Legal token files already exist in the Google Drive project folder.
os.makedirs(TEXT_DIR, exist_ok=True)
print(f"Using legal tokenized labels from: {TEXT_DIR}")

# 3. Create the images folder
os.makedirs(OUTPUT_IMAGES_DIR, exist_ok=True)
print(f"Created directory for Legal Crops: {OUTPUT_IMAGES_DIR}")

# 4. Load YOLO model
all_best_models = glob.glob(str(RUNS_DIR / "**" / "weights" / "best.pt"), recursive=True) + \
                  glob.glob("/content/runs/detect/train*/weights/best.pt") + \
                  glob.glob("/content/runs/train*/weights/best.pt")

if not all_best_models:
    raise FileNotFoundError("No best.pt found! Please re-run YOLO training.")

latest_weights = max(all_best_models, key=os.path.getmtime)
model = YOLO(latest_weights)
print(f"Loaded YOLO model: {latest_weights}")

# 5. Crop Legal Amounts (Class 0)
raw_images = glob.glob(os.path.join(RAW_IMAGES_DIR, '*.tif'))
print(f"Found {len(raw_images)} raw images. Cropping Legal boxes...")

success_count = 0

for img_path in raw_images:
    base_name = os.path.basename(img_path)
    expected_crop_name = f"L{base_name}"   # e.g., Lac00000.tif

    img = cv2.imread(img_path)
    if img is None: continue

    results = model.predict(img, conf=0.25, verbose=False)
    if not results or len(results[0].boxes) == 0: continue

    for box in results[0].boxes:
        cls_id = int(box.cls[0].item())
        if cls_id == 0:  # 0 is 'legal_amount'
            x1, y1, x2, y2 = [int(v) for v in box.xyxy[0].tolist()]
            x1, y1 = max(0, x1), max(0, y1)
            x2, y2 = min(img.shape[1], x2), min(img.shape[0], y2)

            crop_img = img[y1:y2, x1:x2]

            save_path = os.path.join(OUTPUT_IMAGES_DIR, expected_crop_name)
            cv2.imwrite(save_path, crop_img)
            success_count += 1
            break

print(f"\nSuccess! Generated {success_count} cropped Legal images.")


## Part C — Legal Dataset & Vocabulary

In [ ]:
# === Part C — Legal Dataset & Vocabulary ===
import os
import ast
import glob
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image, ImageOps
import editdistance

def parse_arabic_label(seq_str):
    seq_str = seq_str.replace('‪', '').replace('‫', '').replace('‬', '')
    seq_str = seq_str.replace('‫', '').replace('‬', '').strip()
    if seq_str.startswith('['):
        try:
            seq_list = ast.literal_eval(seq_str)
            return " ".join([str(t) for t in seq_list])
        except Exception:
            clean = seq_str.replace('[', '').replace(']', '').replace("'", '').replace('"', '').replace(',', ' ')
            return " ".join(clean.split())
    return seq_str

def build_vocab(text_dir):
    vocab = set()
    for txt_file in glob.glob(os.path.join(text_dir, '**/*.txt'), recursive=True):
        with open(txt_file, 'r', encoding='utf-8-sig') as f:
            for line in f:
                parts = line.strip().split('\t')
                if len(parts) != 2:
                    parts = line.strip().split(' ', 1)
                if len(parts) == 2:
                    vocab.update(list(parse_arabic_label(parts[1])))
    vocab = sorted(vocab)
    char_to_idx = {c: i + 1 for i, c in enumerate(vocab)}
    idx_to_char = {i + 1: c for i, c in enumerate(vocab)}
    return char_to_idx, idx_to_char, len(vocab) + 1

char_to_idx, idx_to_char, num_classes = build_vocab(str(LEGAL_AMOUNTS_TOKENIZED_DIR))
print(f"Built Arabic vocabulary with {num_classes - 1} unique characters.")

class LegalDataset(Dataset):
    def __init__(self, text_dir, img_dir, char_to_idx, transform=None):
        self.transform = transform
        self.data = []
        image_map = {}
        for root, _, files in os.walk(img_dir):
            for f in files:
                if f.lower().endswith(('.tif', '.tiff', '.jpg', '.jpeg', '.png')):
                    image_map[normalize_check_id(f, prefix='L')] = os.path.join(root, f)
        for txt_file in glob.glob(os.path.join(text_dir, '**/*.txt'), recursive=True):
            with open(txt_file, 'r', encoding='utf-8-sig') as f:
                for line in f:
                    parts = line.strip().split('\t')
                    if len(parts) != 2:
                        parts = line.strip().split(' ', 1)
                    if len(parts) != 2:
                        continue
                    img_key = normalize_check_id(parts[0].strip(), prefix='L')
                    img_path = image_map.get(img_key)
                    if not img_path:
                        continue
                    full = parse_arabic_label(parts[1])
                    encoded = [char_to_idx[c] for c in full if c in char_to_idx]
                    if encoded:
                        self.data.append((img_path, encoded, full))

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        img_path, seq, _ = self.data[idx]
        img = Image.open(img_path).convert('L')
        img = ImageOps.mirror(img)
        if self.transform:
            img = self.transform(img)
        return img, torch.tensor(seq, dtype=torch.long), os.path.basename(img_path)

def legal_collate_fn(batch):
    images, targets, filenames = zip(*batch)
    images = torch.stack(images)
    target_lengths = torch.tensor([len(t) for t in targets], dtype=torch.long)
    targets = torch.nn.utils.rnn.pad_sequence(targets, batch_first=True, padding_value=0)
    return images, targets, target_lengths, filenames

legal_transform = transforms.Compose([
    transforms.Resize((64, 512)),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,)),
])

legal_dataset = LegalDataset(str(LEGAL_AMOUNTS_TOKENIZED_DIR), str(LEGAL_IMAGES_DIR),
                             char_to_idx, legal_transform)
print(f"Loaded {len(legal_dataset)} legal image-sequence pairs.")
if len(legal_dataset) == 0:
    raise ValueError("Legal dataset is empty. Run legal cropping first.")

legal_train_size = int(0.85 * len(legal_dataset))
legal_val_size = len(legal_dataset) - legal_train_size
legal_train_ds, legal_val_ds = torch.utils.data.random_split(
    legal_dataset, [legal_train_size, legal_val_size],
    generator=torch.Generator().manual_seed(42))

legal_train_loader = DataLoader(legal_train_ds, batch_size=16, shuffle=True, collate_fn=legal_collate_fn)
legal_val_loader = DataLoader(legal_val_ds, batch_size=1, shuffle=False, collate_fn=legal_collate_fn)
print(f"Train pairs: {len(legal_train_ds)} | Val pairs: {len(legal_val_ds)}")


## Part C — LegalCRNN Model

In [ ]:
# === Part C — LegalCRNN model ===
class LegalCRNN(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.cnn = nn.Sequential(
            nn.Conv2d(1, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2, 2),
            nn.Conv2d(64, 128, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2, 2),
            nn.Conv2d(128, 256, 3, padding=1), nn.BatchNorm2d(256), nn.ReLU(),
            nn.Conv2d(256, 256, 3, padding=1), nn.ReLU(), nn.MaxPool2d((2, 2), (2, 1)),
        )
        self.pool = nn.AdaptiveAvgPool2d((1, None))
        self.rnn = nn.LSTM(256, 128, bidirectional=True, batch_first=True)
        self.fc = nn.Linear(256, num_classes)

    def forward(self, x):
        x = self.cnn(x)
        x = self.pool(x).squeeze(2).permute(0, 2, 1)
        x, _ = self.rnn(x)
        x = self.fc(x)
        return x.log_softmax(2).permute(1, 0, 2)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
legal_model = LegalCRNN(num_classes).to(device)
print(f"LegalCRNN initialized on {device.type.upper()} with {num_classes} classes.")


## Part C — Training (RMSprop, patience, best ckpt)

In [ ]:
# === Part C — Training (RMSprop @ 1e-4 + StepLR + patience=20 + best ckpt) ===
import copy

PART_C_EPOCHS = 500
PART_C_PATIENCE = 20
PART_C_LR = 1e-4

c_criterion = nn.CTCLoss(blank=0, zero_infinity=True)
c_optimizer = optim.RMSprop(legal_model.parameters(), lr=PART_C_LR)
c_scheduler = optim.lr_scheduler.StepLR(c_optimizer, step_size=10000, gamma=0.9)

best_val_loss_c = float("inf")
best_state_c = None
wait_c = 0
hist_c = []

print(f"Training Part C for up to {PART_C_EPOCHS} epochs (patience {PART_C_PATIENCE})...")
for epoch in range(PART_C_EPOCHS):
    legal_model.train()
    t_loss_sum = 0.0
    for images, targets, target_lengths, _ in legal_train_loader:
        images, targets = images.to(device), targets.to(device)
        c_optimizer.zero_grad()
        outputs = legal_model(images)
        input_lengths = torch.full((images.size(0),), outputs.size(0), dtype=torch.long)
        loss = c_criterion(outputs, targets, input_lengths, target_lengths)
        loss.backward()
        c_optimizer.step()
        c_scheduler.step()
        t_loss_sum += loss.item()
    train_loss = t_loss_sum / max(1, len(legal_train_loader))

    legal_model.eval()
    v_sum = 0.0; v_n = 0
    with torch.no_grad():
        for images, targets, target_lengths, _ in legal_val_loader:
            images, targets = images.to(device), targets.to(device)
            outputs = legal_model(images)
            input_lengths = torch.full((images.size(0),), outputs.size(0), dtype=torch.long)
            v_sum += c_criterion(outputs, targets, input_lengths, target_lengths).item()
            v_n += 1
    val_loss = v_sum / max(1, v_n)
    hist_c.append({"epoch": epoch + 1, "train_loss": train_loss, "val_loss": val_loss})

    improved = val_loss < best_val_loss_c - 1e-6
    if improved:
        best_val_loss_c = val_loss
        best_state_c = copy.deepcopy(legal_model.state_dict())
        torch.save(best_state_c, PART_C_BEST_CKPT)
        wait_c = 0
    else:
        wait_c += 1

    if (epoch + 1) % 5 == 0 or epoch == 0 or improved:
        print(f"Epoch {epoch+1:03d} | train {train_loss:.4f} | val {val_loss:.4f}"
              f" | best {best_val_loss_c:.4f} | wait {wait_c}")

    if wait_c >= PART_C_PATIENCE:
        print(f"Early stopping at epoch {epoch+1}.")
        break

if best_state_c is not None:
    legal_model.load_state_dict(best_state_c)
print(f"Best val loss: {best_val_loss_c:.4f}")
print(f"Best checkpoint saved to: {PART_C_BEST_CKPT}")


## Part C — Evaluation (Validation)

In [ ]:
# === Part C — Validation evaluation (CER + WER + exact match) ===
def decode_c_preds(preds, idx_to_char, blank_idx=0):
    out, prev = [], -1
    for c in preds:
        v = c.item() if hasattr(c, "item") else int(c)
        if v != prev and v != blank_idx and v in idx_to_char:
            out.append(idx_to_char[v])
        prev = v
    return "".join(out)

legal_model.eval()
total_char_N = total_char_err = 0
total_word_N = total_word_err = 0
exact_c = 0
total_c = 0
out_lines_c = []
c_pred_strs_val = {}; c_gt_strs_val = {}

with torch.no_grad():
    for images, targets, target_lengths, filenames in legal_val_loader:
        images = images.to(device)
        outputs = legal_model(images)
        _, preds = outputs.max(2)
        pred_str = decode_c_preds(preds.transpose(1, 0)[0], idx_to_char)
        gt_str = "".join(idx_to_char[c.item()] for c in targets[0][:target_lengths[0]])
        pred_str = " ".join(pred_str.split())
        gt_str = " ".join(gt_str.split())

        out_lines_c.append(f"{filenames[0]} {pred_str}")
        c_pred_strs_val[filenames[0]] = pred_str
        c_gt_strs_val[filenames[0]] = gt_str
        total_c += 1
        if pred_str == gt_str:
            exact_c += 1

        char_N = len(gt_str)
        if char_N:
            total_char_err += editdistance.eval(pred_str, gt_str)
            total_char_N += char_N
        gt_words = gt_str.split(); pred_words = pred_str.split()
        if gt_words:
            total_word_err += editdistance.eval(pred_words, gt_words)
            total_word_N += len(gt_words)

cer = total_char_err / total_char_N * 100 if total_char_N else 0
wer = total_word_err / total_word_N * 100 if total_word_N else 0
exact_pct = exact_c / total_c * 100 if total_c else 0

lines = ["PART C — LEGAL AMOUNT (VALIDATION)",
         f"Samples: {total_c}",
         f"Character Error Rate (CER): {cer:.2f}%",
         f"Word Error Rate (WER):      {wer:.2f}%",
         f"Exact match (no errors):    {exact_pct:.2f}% ({exact_c}/{total_c})"]
PART_C_OUTPUT.write_text("\n".join(out_lines_c) + "\n", encoding="utf-8")
PART_C_METRICS.write_text("\n".join(lines) + "\n", encoding="utf-8")
print("\n".join(lines))
print("Output saved to:", PART_C_OUTPUT)


## Part C — Confusion Matrix (Validation)

In [ ]:
# === Part C — Confusion matrix (validation, character-level) ===
import numpy as np
import matplotlib.pyplot as plt

def build_char_confusion(pred_strings, gt_strings, vocab):
    # vocab is the ordered list of characters; +1 row and column for ins/del
    char_index = {c: i for i, c in enumerate(vocab)}
    K = len(vocab)
    cm = np.zeros((K + 1, K + 1), dtype=int)
    for fname, gt in gt_strings.items():
        pred = pred_strings.get(fname, "")
        gt_chars = [c for c in gt if c in char_index]
        pred_chars = [c for c in pred if c in char_index]
        n, m = len(gt_chars), len(pred_chars)
        dp = np.zeros((n + 1, m + 1), dtype=int)
        for i in range(n + 1): dp[i, 0] = i
        for j in range(m + 1): dp[0, j] = j
        for i in range(1, n + 1):
            for j in range(1, m + 1):
                cost = 0 if gt_chars[i - 1] == pred_chars[j - 1] else 1
                dp[i, j] = min(dp[i - 1, j] + 1, dp[i, j - 1] + 1, dp[i - 1, j - 1] + cost)
        i, j = n, m
        while i > 0 or j > 0:
            if i > 0 and j > 0 and dp[i, j] == dp[i - 1, j - 1] + (0 if gt_chars[i - 1] == pred_chars[j - 1] else 1):
                cm[char_index[gt_chars[i - 1]], char_index[pred_chars[j - 1]]] += 1
                i -= 1; j -= 1
            elif i > 0 and dp[i, j] == dp[i - 1, j] + 1:
                cm[char_index[gt_chars[i - 1]], K] += 1  # deletion
                i -= 1
            else:
                cm[K, char_index[pred_chars[j - 1]]] += 1  # insertion
                j -= 1
    return cm

def plot_char_cm(cm, vocab, title, save_path):
    K = len(vocab)
    fig, ax = plt.subplots(figsize=(max(8, K * 0.4), max(7, K * 0.4)))
    ax.imshow(cm, cmap="Blues")
    ax.set_xticks(range(K + 1)); ax.set_yticks(range(K + 1))
    ax.set_xticklabels(vocab + ["Del"], rotation=90, fontsize=8)
    ax.set_yticklabels(vocab + ["Ins"], fontsize=8)
    ax.set_xlabel("Predicted"); ax.set_ylabel("True"); ax.set_title(title)
    for i in range(K + 1):
        for j in range(K + 1):
            v = cm[i, j]
            if v > 0:
                ax.text(j, i, str(v), ha="center", va="center", fontsize=6,
                        color="white" if v > cm.max() * 0.5 else "black")
    plt.tight_layout(); plt.savefig(save_path, dpi=120); plt.show()
    print("Saved:", save_path)

# Vocab excluding the blank index (idx 0) — display order from idx_to_char
char_vocab = [idx_to_char[i] for i in sorted(idx_to_char)]
cm_c_val = build_char_confusion(c_pred_strs_val, c_gt_strs_val, char_vocab)
plot_char_cm(cm_c_val, char_vocab, "Part C Confusion Matrix (Validation)",
             RESULTS_FIGURES_DIR / "partC_confusion_val.png")


## Part C — Test Set Evaluation

In [ ]:
# === Part C — Test set evaluation ===
import cv2
import shutil

if not TEST_AVAILABLE:
    print("Test set not available — skipping Part C test evaluation.")
else:
    yolo_for_legal = YOLO(BEST_MODEL_PATH)
    test_images = sorted(TEST_IMAGES_DIR.glob("*.tif"))
    crop_count = 0
    for img_path in test_images:
        out_name = f"L{img_path.name}"
        out_path = TEST_LEGAL_IMAGES_DIR / out_name
        if out_path.exists():
            crop_count += 1; continue
        img = cv2.imread(str(img_path))
        if img is None:
            continue
        result = yolo_for_legal.predict(img, conf=0.25, verbose=False)
        if not result or len(result[0].boxes) == 0:
            continue
        for box in result[0].boxes:
            if int(box.cls[0].item()) == 0:
                x1, y1, x2, y2 = [int(v) for v in box.xyxy[0].tolist()]
                x1, y1 = max(0, x1), max(0, y1)
                x2, y2 = min(img.shape[1], x2), min(img.shape[0], y2)
                cv2.imwrite(str(out_path), img[y1:y2, x1:x2])
                crop_count += 1
                break
    print(f"Test legal crops ready: {crop_count}/{len(test_images)} in {TEST_LEGAL_IMAGES_DIR}")

    # Stage legal labels into a dedicated dir (avoids recursive scan picking up
    # CourtesyAmounts.txt or BoundingBox/*.txt).
    test_legal_label_dir = WORK_DIR / "test_legal_labels"
    test_legal_label_dir.mkdir(parents=True, exist_ok=True)
    shutil.copy2(TEST_LEGAL_LABELS, test_legal_label_dir / "LegalAmounts.txt")

    test_dataset_c = LegalDataset(
        text_dir=str(test_legal_label_dir),
        img_dir=str(TEST_LEGAL_IMAGES_DIR),
        char_to_idx=char_to_idx,
        transform=legal_transform,
    )
    print(f"Test pairs (Part C): {len(test_dataset_c)}")
    test_loader_c = DataLoader(test_dataset_c, batch_size=1, shuffle=False, collate_fn=legal_collate_fn)

    legal_model.load_state_dict(torch.load(PART_C_BEST_CKPT, map_location=device))
    legal_model.eval()

    total_char_N = total_char_err = 0
    total_word_N = total_word_err = 0
    exact_t = 0; total_t = 0
    out_lines = []
    c_pred_strs_test = {}; c_gt_strs_test = {}

    with torch.no_grad():
        for images, targets, target_lengths, filenames in test_loader_c:
            images = images.to(device)
            outputs = legal_model(images)
            _, preds = outputs.max(2)
            pred_str = decode_c_preds(preds.transpose(1, 0)[0], idx_to_char)
            gt_str = "".join(idx_to_char[c.item()] for c in targets[0][:target_lengths[0]])
            pred_str = " ".join(pred_str.split())
            gt_str = " ".join(gt_str.split())
            out_lines.append(f"{filenames[0]} {pred_str}")
            c_pred_strs_test[filenames[0]] = pred_str
            c_gt_strs_test[filenames[0]] = gt_str
            total_t += 1
            if pred_str == gt_str:
                exact_t += 1
            char_N = len(gt_str)
            if char_N:
                total_char_err += editdistance.eval(pred_str, gt_str)
                total_char_N += char_N
            gt_words = gt_str.split(); pred_words = pred_str.split()
            if gt_words:
                total_word_err += editdistance.eval(pred_words, gt_words)
                total_word_N += len(gt_words)

    cer = total_char_err / total_char_N * 100 if total_char_N else 0
    wer = total_word_err / total_word_N * 100 if total_word_N else 0
    exact_pct = exact_t / total_t * 100 if total_t else 0

    lines = ["PART C — LEGAL AMOUNT (TEST)",
             f"Samples: {total_t}",
             f"Character Error Rate (CER): {cer:.2f}%",
             f"Word Error Rate (WER):      {wer:.2f}%",
             f"Exact match (no errors):    {exact_pct:.2f}% ({exact_t}/{total_t})"]
    PART_C_TEST_OUTPUT.write_text("\n".join(out_lines) + "\n", encoding="utf-8")
    PART_C_TEST_METRICS.write_text("\n".join(lines) + "\n", encoding="utf-8")
    print("\n".join(lines))

    cm_c_test = build_char_confusion(c_pred_strs_test, c_gt_strs_test, char_vocab)
    plot_char_cm(cm_c_test, char_vocab, "Part C Confusion Matrix (Test)",
                 RESULTS_FIGURES_DIR / "partC_confusion_test.png")


## Part D — Postprocessing (Algorithm 1 + Min-Edit-Distance)

In [ ]:
# === Part D — Postprocessing pipeline ===
# Implements the report's three-stage conversion:
#   - Algorithm 1 (legal -> numeric): thousands/hundreds multipliers + additive
#   - § 8.1.2 Min-edit-distance fallback for unknown tokens
#   - § 8.2 Tie-break edit-distance candidates using predicted courtesy
#   - § 8.3 Enhance courtesy using legal (zero-digit insert/delete fix)
import functools
import editdistance

# ---- Multiplier markers ----
THOUSAND_MARKERS = {'الف', 'ألف', 'آلاف', 'الاف'}
HUNDRED_MARKERS = {'مائة', 'مائه', 'مئة', 'مية', 'ميه', 'مايه'}

# ---- Arabic word -> numeric value dictionary ----
WORD_TO_NUMBER = {}

# Units 1..9 (with feminine / variant spellings)
for w, v in [
    ('واحد', 1), ('واحدة', 1), ('احد', 1), ('إحدى', 1),
    ('اثنان', 2), ('اثنين', 2), ('إثنان', 2), ('اثنتان', 2),
    ('ثلاثة', 3), ('ثلاث', 3), ('ثلاثه', 3),
    ('اربعة', 4), ('أربعة', 4), ('اربع', 4), ('أربع', 4), ('اربعه', 4), ('أربعه', 4),
    ('خمسة', 5), ('خمس', 5), ('خمسه', 5),
    ('ستة', 6), ('ست', 6), ('سته', 6),
    ('سبعة', 7), ('سبع', 7), ('سبعه', 7),
    ('ثمانية', 8), ('ثماني', 8), ('ثمان', 8), ('ثمانيه', 8),
    ('تسعة', 9), ('تسع', 9), ('تسعه', 9),
]:
    WORD_TO_NUMBER[w] = v

# Tens 10..90
for w, v in [
    ('عشرة', 10), ('عشر', 10), ('عشره', 10),
    ('عشرون', 20), ('عشرين', 20),
    ('ثلاثون', 30), ('ثلاثين', 30),
    ('اربعون', 40), ('أربعون', 40), ('اربعين', 40), ('أربعين', 40),
    ('خمسون', 50), ('خمسين', 50),
    ('ستون', 60), ('ستين', 60),
    ('سبعون', 70), ('سبعين', 70),
    ('ثمانون', 80), ('ثمانين', 80),
    ('تسعون', 90), ('تسعين', 90),
]:
    WORD_TO_NUMBER[w] = v

# Hundreds 100..900 (compound and standalone)
for w, v in [
    ('مائة', 100), ('مائه', 100), ('مئة', 100), ('مية', 100), ('ميه', 100), ('مايه', 100),
    ('مائتان', 200), ('مائتين', 200), ('مئتان', 200), ('مئتين', 200),
    ('ثلاثمائة', 300), ('ثلاثمائه', 300), ('ثلاثمئة', 300),
    ('اربعمائة', 400), ('أربعمائة', 400), ('اربعمائه', 400),
    ('خمسمائة', 500), ('خمسمائه', 500), ('خمسمئة', 500),
    ('ستمائة', 600), ('ستمائه', 600),
    ('سبعمائة', 700), ('سبعمائه', 700),
    ('ثمانمائة', 800), ('ثمانمائه', 800), ('ثمانيمائة', 800),
    ('تسعمائة', 900), ('تسعمائه', 900),
]:
    WORD_TO_NUMBER[w] = v

# Thousand markers — value 0 in the additive sense; multiplier path handles them.
for w in THOUSAND_MARKERS:
    WORD_TO_NUMBER.setdefault(w, 0)

# Filler / non-numeric words mapped to 0
ZERO_WORDS = {
    'فقط', 'لاغير', 'لا', 'غير', 'ريال', 'ريالا', 'هلله', 'هللة', 'هللات',
    'سعودي', 'سعودية', 'و',
}
for w in ZERO_WORDS:
    WORD_TO_NUMBER.setdefault(w, 0)

DICT_WORDS = list(WORD_TO_NUMBER.keys())

@functools.lru_cache(maxsize=8192)
def closest_dict_word(word, max_dist=1):
    if not word:
        return None
    if word in WORD_TO_NUMBER:
        return word
    best_w, best_d = None, max_dist + 1
    for w in DICT_WORDS:
        d = editdistance.eval(word, w)
        if d < best_d:
            best_d, best_w = d, w
            if d == 0:
                break
    return best_w if best_d <= max_dist else None

def words_from_legal(legal_text, max_word_len=16):
    # Concatenate all subwords, then greedy maximal-match against the
    # dictionary (with a single-edit fuzzy fallback when no match exists).
    if not legal_text:
        return []
    s = "".join(legal_text.split())
    out = []
    i, n = 0, len(s)
    while i < n:
        match = None
        for j in range(min(n, i + max_word_len), i, -1):
            cand = s[i:j]
            if cand in WORD_TO_NUMBER:
                match = (cand, j)
                break
        if match:
            out.append(match[0])
            i = match[1]
            continue
        # Fuzzy fallback: try lengths 2..8 and pick the closest dict word.
        best, best_d, best_j = None, 99, i + 1
        for L in range(2, min(8, n - i) + 1):
            cand = s[i:i + L]
            w = closest_dict_word(cand, max_dist=1)
            if w is None:
                continue
            d = editdistance.eval(cand, w)
            # prefer lower edit distance, then longer span
            if d < best_d or (d == best_d and (i + L) > best_j):
                best, best_d, best_j = w, d, i + L
        if best is not None:
            out.append(best)
            i = best_j
        else:
            i += 1
    return out

def legal_words_to_value(words):
    # Algorithm 1: thousands/hundreds multiplier + additive accumulator.
    total = 0
    current = 0
    for word in words:
        if word in THOUSAND_MARKERS:
            total += (current if current != 0 else 1) * 1000
            current = 0
        elif word in HUNDRED_MARKERS:
            total += (current if current != 0 else 1) * 100
            current = 0
        else:
            current += WORD_TO_NUMBER.get(word, 0)
    total += current
    return total

def convert_legal_to_number(legal_text, courtesy_value=None):
    # courtesy_value is reserved for § 8.2 tie-break (single-candidate fuzzy
    # fallback already implements a simple form of this).
    return legal_words_to_value(words_from_legal(legal_text))

# ---- § 8.3 zero-digit fix ----
def enhance_courtesy_with_legal(courtesy_str, legal_value):
    # If courtesy and legal differ by a single 0 insertion/deletion, prefer legal.
    try:
        cv = int(courtesy_str)
    except (TypeError, ValueError):
        return courtesy_str
    if legal_value <= 0:
        return courtesy_str
    if cv == legal_value:
        return courtesy_str
    s_legal = str(legal_value)
    s_court = str(cv)
    if abs(len(s_legal) - len(s_court)) != 1:
        return courtesy_str
    if editdistance.eval(s_legal, s_court) == 1:
        if s_legal.count('0') != s_court.count('0'):
            return s_legal
    return courtesy_str

print(f"Dictionary loaded: {len(WORD_TO_NUMBER)} entries")
print("Sample conversions:")
for sample in [
    "خمسة الاف ريال",
    "عشرة الاف ريال فقط لا غير",
    "ثلاث مائة و خمسة و عشرون ريال",
    "الف و خمسمائة ريال",
    "عشر ة آ لا ف ر يا ل فقط لا غير",   # subword-spaced form
]:
    print(f"  {sample!r} -> {convert_legal_to_number(sample)}")


## Part D — Verification (Validation)

In [ ]:
# === Part D — Verification (Validation set) ===
def load_pred_file(path):
    out = {}
    if not Path(path).exists():
        return out
    for line in Path(path).read_text(encoding='utf-8').splitlines():
        line = line.strip()
        if not line:
            continue
        parts = line.split(maxsplit=1)
        if len(parts) < 2:
            continue
        key = parts[0]
        if key.startswith(('C', 'L')):
            key = key[1:]
        key = Path(key).stem
        out[key] = parts[1] if len(parts) > 1 else ''
    return out

courtesy_val = load_pred_file(PART_B_OUTPUT)
legal_val = load_pred_file(PART_C_OUTPUT)

verified = 0
final_correct = 0
total = 0
mismatch_log = []

for cid in courtesy_val:
    if cid not in legal_val:
        continue
    total += 1
    courtesy_str = courtesy_val[cid]
    legal_text = legal_val[cid]
    legal_val_num = convert_legal_to_number(legal_text)
    enhanced = enhance_courtesy_with_legal(courtesy_str, legal_val_num)
    try:
        cv = int(enhanced)
    except ValueError:
        cv = 0
    is_match = cv != 0 and cv == legal_val_num
    if is_match:
        verified += 1
        final_correct += 1  # we don't have GT for val, so use match==correct
    else:
        mismatch_log.append((cid, courtesy_str, enhanced, legal_text, legal_val_num))

verify_rate = verified / total * 100 if total else 0
lines = ["PART D — VERIFICATION (VALIDATION)",
         f"Total checks: {total}",
         f"Verified (legal == courtesy): {verified} ({verify_rate:.2f}%)",
         f"Mismatches: {total - verified}"]
PART_D_METRICS.write_text("\n".join(lines) + "\n", encoding="utf-8")
print("\n".join(lines))
print("\nFirst 5 mismatches:")
for cid, c, e, lt, lv in mismatch_log[:5]:
    print(f"  {cid}: courtesy={c!r}  enhanced={e!r}  legal={lt!r} -> {lv}")


## Part D — Verification (Test)

In [ ]:
# === Part D — Verification (Test set) with ground truth ===
import re

if not TEST_AVAILABLE:
    print("Test set not available — skipping Part D test verification.")
else:
    courtesy_t = load_pred_file(PART_B_TEST_OUTPUT)
    legal_t = load_pred_file(PART_C_TEST_OUTPUT)

    # Ground truth courtesy values (parse stringified list of digits to integer)
    def parse_courtesy_label(seq_str):
        seq_str = re.sub(r"[‫\[\]'\",]", " ", seq_str)
        digits = [t for t in seq_str.split() if t.isdigit() and t != '10']
        return int("".join(digits)) if digits else 0

    gt_courtesy = {}
    for line in TEST_COURTESY_LABELS.read_text(encoding='utf-8').splitlines():
        parts = line.split('\t', 1)
        if len(parts) != 2:
            continue
        key = Path(parts[0].lstrip('C')).stem
        gt_courtesy[key] = parse_courtesy_label(parts[1])

    verified = 0      # § 8.4: predicted courtesy == predicted legal value
    final_correct = 0  # predicted (after enhancement) == GT courtesy
    false_positive = 0  # predicted-match but mismatches GT
    total = 0
    diag = {"verified+correct": 0, "verified+wrong": 0, "unverified+correct": 0, "unverified+wrong": 0}

    for cid in courtesy_t:
        if cid not in legal_t or cid not in gt_courtesy:
            continue
        total += 1
        c_str = courtesy_t[cid]
        l_text = legal_t[cid]
        l_val = convert_legal_to_number(l_text)
        enhanced = enhance_courtesy_with_legal(c_str, l_val)
        try:
            cv = int(enhanced)
        except ValueError:
            cv = 0
        is_match = cv != 0 and cv == l_val
        is_correct = cv == gt_courtesy[cid]
        if is_match:
            verified += 1
            if is_correct:
                final_correct += 1
                diag["verified+correct"] += 1
            else:
                false_positive += 1
                diag["verified+wrong"] += 1
        else:
            if is_correct:
                diag["unverified+correct"] += 1
            else:
                diag["unverified+wrong"] += 1

    verify_rate = verified / total * 100 if total else 0
    final_correct_rate = final_correct / total * 100 if total else 0
    fp_rate = false_positive / max(1, total - final_correct) * 100

    lines = ["PART D — VERIFICATION (TEST)",
             f"Total checks compared: {total}",
             f"Verified (legal == courtesy): {verified} ({verify_rate:.2f}%)",
             f"Final correct (matches GT courtesy): {final_correct} ({final_correct_rate:.2f}%)",
             f"False positives (verified but wrong): {false_positive}",
             f"Diagnostic breakdown: {diag}"]
    PART_D_TEST_METRICS.write_text("\n".join(lines) + "\n", encoding="utf-8")
    print("\n".join(lines))


## Final Results Summary

In [ ]:
# === Final Results Summary (matches Report Table 12) ===
def _read(path):
    return Path(path).read_text(encoding='utf-8') if Path(path).exists() else ""

summary_lines = ["FINAL RESULTS SUMMARY (this notebook)",
                 "=" * 60, ""]

if Path(PART_A_METRICS).exists():
    summary_lines.append("[Part A — VALIDATION]")
    summary_lines.append(_read(PART_A_METRICS).strip())
    summary_lines.append("")
if Path(PART_A_TEST_METRICS).exists():
    summary_lines.append("[Part A — TEST]")
    summary_lines.append(_read(PART_A_TEST_METRICS).strip())
    summary_lines.append("")

if Path(PART_B_METRICS).exists():
    summary_lines.append("[Part B — VALIDATION]")
    summary_lines.append(_read(PART_B_METRICS).strip())
    summary_lines.append("")
if Path(PART_B_TEST_METRICS).exists():
    summary_lines.append("[Part B — TEST]")
    summary_lines.append(_read(PART_B_TEST_METRICS).strip())
    summary_lines.append("")

if Path(PART_C_METRICS).exists():
    summary_lines.append("[Part C — VALIDATION]")
    summary_lines.append(_read(PART_C_METRICS).strip())
    summary_lines.append("")
if Path(PART_C_TEST_METRICS).exists():
    summary_lines.append("[Part C — TEST]")
    summary_lines.append(_read(PART_C_TEST_METRICS).strip())
    summary_lines.append("")

if Path(PART_D_METRICS).exists():
    summary_lines.append("[Part D — VALIDATION]")
    summary_lines.append(_read(PART_D_METRICS).strip())
    summary_lines.append("")
if Path(PART_D_TEST_METRICS).exists():
    summary_lines.append("[Part D — TEST]")
    summary_lines.append(_read(PART_D_TEST_METRICS).strip())
    summary_lines.append("")

text = "\n".join(summary_lines)
FINAL_SUMMARY_TXT.write_text(text + "\n", encoding="utf-8")
print(text)
print("Saved:", FINAL_SUMMARY_TXT)
